# Install Dependencies

In [ ]:
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.5/31.5 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.3/366.3 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 34.5 MB/s eta 0:00:00


In [ ]:
!pip install -q datasets
!pip install -q regex

In [ ]:
!pip install -q emoji
!pip install -q PyArabic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 3.5 MB/s eta 0:00:00


In [ ]:
!pip install -q diffusers

# login

In [ ]:
import huggingface_hub
huggingface_hub.login('HF_TOKEN')

# Import Required Modules

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ['CUDA_LAUNCH_BLOCKING']="1"
os.environ['TORCH_USE_CUDA_DSA'] = "1"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import pandas as pd
import random
from sklearn.utils import shuffle
import os
import re
from tqdm import tqdm
import bitsandbytes as bnb
import torch
import torch.nn as nn
import transformers
from datasets import Dataset
from peft import LoraConfig, PeftConfig
from trl import SFTTrainer
from transformers import (AutoModelForCausalLM,
                          AutoTokenizer,
                          BitsAndBytesConfig,
                          TrainingArguments,
                          pipeline,
                          logging)
from sklearn.metrics import (accuracy_score,
                             classification_report,
                             precision_score,
                             recall_score,
                             f1_score,
                             confusion_matrix)
from sklearn.model_selection import train_test_split
import emoji
import pyarabic.araby as araby

In [ ]:
import pandas as pd

In [ ]:
import torch
import torch.distributed as dist

# Load Model

In [ ]:
model_name = "ALLaM-AI/ALLaM-7B-Instruct-preview"

compute_dtype = getattr(torch, "float16")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    # quantization_config=bnb_config,
    device_map={"": 0},
    trust_remote_code=True,
)

model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(model_name,
                                          trust_remote_code=True,
                                          padding_side="left",
                                          add_eos_token=True,
                                         )

# Assign pad_token if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.03G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

In [ ]:
pipe = pipeline(task="text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=20,
                temperature=0.2
               )

Device set to use cuda:0


# Zero Shot

In [ ]:
import pandas as pd
data = pd.read_excel('Physics-QA-ArabicPrompt-Zero Shot.xlsx')

In [ ]:
data.shape

(200, 1)

# Predict

In [ ]:
pred = []

max_length = tokenizer.model_max_length
for i in tqdm(range(len(data))):
    prompt = data.iloc[i]["prompt"]
    result = pipe(prompt[:tokenizer.model_max_length], truncation=True, pad_token_id=pipe.tokenizer.eos_token_id)
    answer = result[0]['generated_text'].split("الإجابة:")[-1].strip()

    pred.append(answer)

100%|██████████| 200/200 [03:03<00:00,  1.09it/s]


In [ ]:
pred_zero = pd.DataFrame()
pred_zero['Predicted'] = pred
pred_zero['Predicted'].value_counts()

,count
Predicted,
أ) 2 ك ع,2
أ) تزداد,2
ب) 10 م/ث,2
ج) 2 ك ع,2
أ) كمية تحرك السيارة = كمية تحرك الشاحنة.,2
...,...
ب) 20 كغم.م/ث,1
ج) نصف القطر,1
أ) مساو لتسارع الرصاصة,1


In [ ]:
nor_pre = []
for pr in pred_zero['Predicted']:
  if "أ)" in pr:
    nor_pre.append("A")
  elif "أ / أ" in pr:
    nor_pre.append("A")
  elif "أ / ب / ج / د)" in pr:
    nor_pre.append("Unclassified")
  elif "ب)" in pr:
    nor_pre.append("B")
  elif "ب" in pr:
    nor_pre.append("B")
  elif "ج)" in pr:
    nor_pre.append("C")
  elif "ج" in pr:
    nor_pre.append("C")
  elif "د)" in pr:
    nor_pre.append("D")

In [ ]:
pred_zero['Normalized Prediction'] = nor_pre

In [ ]:
pred_zero['Normalized Prediction'].value_counts()

,count
Normalized Prediction,
B,68
A,55
C,51
D,26


In [ ]:
pred_zero['prompt'] = data['prompt']
pred_zero.head()

,Predicted,Normalized Prediction,prompt
0,أ) 0.5 م,A,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...
1,د) من Hz 20 إلى Hz 20000,D,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...
2,أ) 1.7 نيوتن,A,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...
3,د) 10 كلفن,D,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...
4,أ) 2 ك ع,A,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...


In [ ]:
pred_zero.to_excel('Allam Zero Shot QA Physics.xlsx', index = False)

In [ ]:
true = pd.read_excel('sample_physics.xlsx')
y_true = true['Answer Key'].values
print(classification_report(y_true, pred_zero['Normalized Prediction'].values, digits = 4))

              precision    recall  f1-score   support

           A     0.4909    0.5510    0.5192        49
           B     0.4559    0.5345    0.4921        58
           C     0.4902    0.5952    0.5376        42
           D     0.5769    0.2941    0.3896        51

    accuracy                         0.4900       200
   macro avg     0.5035    0.4937    0.4846       200
weighted avg     0.5025    0.4900    0.4822       200



# Pred Few Shot

In [ ]:
data2 = pd.read_excel('Physics-QA-ArabicPrompt-Few Shot.xlsx')

In [ ]:
pred = []

max_length = tokenizer.model_max_length
for i in tqdm(range(len(data2))):
    prompt = data2.iloc[i]["prompt"]
    result = pipe(prompt[:tokenizer.model_max_length], truncation=True, pad_token_id=pipe.tokenizer.eos_token_id)
    answer = result[0]['generated_text'].split("الإجابة:")[-1].strip()

    pred.append(answer)

100%|██████████| 200/200 [03:41<00:00,  1.11s/it]


In [ ]:
pred_few = pd.DataFrame()
pred_few['Predicted'] = pred
pred_few['Predicted'].value_counts()

,count
Predicted,
ب) صفر,3
ب) 10 م/ث,2
ج) 100,2
ج) 2 ك ع,2
د) 0.8,2
...,...
ب) 20 كغم.م/ث,1
ج) نصف القطر,1
أ) مساو لتسارع الرصاصة,1


In [ ]:
nor_pre = []
for pr in pred_few['Predicted']:
  if "أ)" in pr:
    nor_pre.append("A")
  elif "أ / أ" in pr:
    nor_pre.append("A")
  elif "أ / ب / ج / د)" in pr:
    nor_pre.append("Unclassified")
  elif "ب)" in pr:
    nor_pre.append("B")
  elif "ب" in pr:
    nor_pre.append("B")
  elif "ج)" in pr:
    nor_pre.append("C")
  elif "ج" in pr:
    nor_pre.append("C")
  elif "د)" in pr:
    nor_pre.append("D")

In [ ]:
pred_few['Normalized Prediction'] = nor_pre
pred_few['Normalized Prediction'].value_counts()

,count
Normalized Prediction,
B,77
C,58
A,34
D,31


In [ ]:
pred_few['prompt'] = data2['prompt']
pred_few.head()

,Predicted,Normalized Prediction,prompt
0,أ) 0.5 م,A,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...
1,د) من Hz 20 إلى Hz 20000,D,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...
2,ج) 5.1 نيوتن,C,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...
3,ب) 5 كلفن,B,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...
4,ب) صفر,B,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...


In [ ]:
pred_few.to_csv('Allam Few Shot QA Physics.xlsx', index = False)

In [ ]:
print(classification_report(y_true, pred_few['Normalized Prediction'].values, digits = 4))

              precision    recall  f1-score   support

           A     0.5294    0.3673    0.4337        49
           B     0.4805    0.6379    0.5481        58
           C     0.4310    0.5952    0.5000        42
           D     0.4839    0.2941    0.3659        51

    accuracy                         0.4750       200
   macro avg     0.4812    0.4737    0.4619       200
weighted avg     0.4830    0.4750    0.4635       200



# CoT

In [ ]:
pipe = pipeline(task="text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=50,
                temperature=0.2
               )

Device set to use cuda:0


In [ ]:
data3 = pd.read_excel('Physics-QA-ArabicPrompt-CoT.xlsx')

In [ ]:
pred = []

max_length = tokenizer.model_max_length
for i in tqdm(range(len(data3))):
    prompt = data3.iloc[i]["prompt"]
    result = pipe(prompt[:tokenizer.model_max_length], truncation=True, pad_token_id=pipe.tokenizer.eos_token_id)
    answer = result[0]['generated_text'].split("السؤال الذي يجب عليك الإجابة عليه:")[-1].strip()

    pred.append(answer)

100%|██████████| 200/200 [06:53<00:00,  2.07s/it]


In [ ]:
pred_cot = pd.DataFrame()
pred_cot['Predicted'] = pred
pred_cot['Predicted'].value_counts()

,count
Predicted,
السؤال:\nمقاومتان مقدار كل منهما م إذا وصلتا على التوازي كافأتا\n الخيارات:\nأ) 0.5 م\nب) م\nج) 2 م\nد) 1.5 م \nالإجابة:\nأ) 0.5 م,1
السؤال:\nما هو مدى الموجات الصوت ةٌ الت تٌمكن الإنسان السل مٌ من سماعها ؟\n الخيارات:\nأ) أكبر من Hz 20000\nب) من Hz 10 إلى Hz 10000\nج) أقل من Hz 20\nد) من Hz 20 إلى Hz 20000 \nالإجابة:\nب) من Hz 10 إلى Hz 100,1
السؤال:\n جسم كتلته 5 كغم وكمية تحركه 15 كغم.م/ث ، فإن مقدار محصلة القوى الالزمة لزيادة سرعة الجسم إلى 8 م/ث\nخالل 15 ثانية هو :\n\n الخيارات:\nأ) 1.7 نيوتن\nب) 0.35 نيوتن\nج) 5.1 نيوتن\nد) 11 نيوتن \nالإجابة:\nج) 5.1 نيوتن,1
السؤال:\nالفرق في درجات الحرارة الالزمة لمادة معاملها الحراري - 0.05 / ك إلنقاص مقاوميتها للنصف\n الخيارات:\nأ) 20 كلفن\nب) 5 كلفن\nج) 12 كلفن\nد) 10 كلفن \nالإجابة:\nب) 5 كلفن,1
"السؤال:\n كمية التحرك للنظام الذي يتكون من كرتين متماثلتين في الكتلة "" ك "" وتسيران باتجاهين متعاكسين و بنفس السرعة "" ع "" هي\n الخيارات:\nأ) 2 ك ع\nب) صفر\nج) ك ع\nد) 0.5 ك ع \nالإجابة:\nب) صفر",1
...,...
السؤال:\n جسم كمية تحركه 10 كغم.م/ث فإذا ضاعفنا طاقته الحركية فإن كت تساوي :\n\n الخيارات:\nأ) 5 كغم.م/ث\nب) 20 كغم.م/ث\nج) 10 كغم.م/ث\nد) 10 كغم.م/ث \nالإجابة:\nب) 20 كغم.م/ث,1
السؤال:\nتتناسب شدة المجال عند مركز الملف الدائري عكسيا مع\n الخيارات:\nأ) النفاذية المغناطيسية\nب) شدة التيار\nج) نصف القطر\nد) عدد اللفات \nالإجابة:\nج) نصف القطر,1
"السؤال:\n يحمل صياد بندقية صيد ويطلق رصاصة باتجاه هدف متحرك، و بناءاً على قانون نيوتن الثالث تنطلق الرصاصة إلى الأمام فيما يرتد الصياد و البندقية إلى الخلف ,وعليه يجب أن يكون تسارع البندقية و الرجل :\n الخيارات:\nأ) مساو لتسارع الرصاصة\nب) أكبر من تسارع الرصاصة\nج) صفر\nد) أقل من تسارع الرصاصة \nالإجابة:\nأ) مساو لتسارع الرصاصة",1


In [ ]:
nor_pre = []
for pr in pred_cot['Predicted']:
  if "الإجابة:" in pr:
    answer = pr.split("الإجابة:")[-1].strip()
    if "أ)" in answer:
      nor_pre.append("A")
    elif "أ / أ" in answer:
      nor_pre.append("A")
    elif "أ / ب / ج / د)" in answer:
      nor_pre.append("Unclassified")
    elif "ب)" in answer:
      nor_pre.append("B")
    elif "ب" in answer:
      nor_pre.append("B")
    elif "ج)" in answer:
      nor_pre.append("C")
    elif "ج" in answer:
      nor_pre.append("C")
    elif "د)" in answer:
      nor_pre.append("D")
  elif "الإجابة النهائية:" in pr:
    answer = pr.split("الإجابة النهائية:")[-1].strip()
    if "أ)" in answer:
      nor_pre.append("A")
    elif "أ / أ" in answer:
      nor_pre.append("A")
    elif "أ / ب / ج / د)" in answer:
      nor_pre.append("Unclassified")
    elif "ب)" in answer:
      nor_pre.append("B")
    elif "ب" in answer:
      nor_pre.append("B")
    elif "ج)" in answer:
      nor_pre.append("C")
    elif "ج" in answer:
      nor_pre.append("C")
    elif "د)" in answer:
      nor_pre.append("D")
  else:
    nor_pre.append("A")

In [ ]:
pred_cot['Normalized Prediction'] = nor_pre
pred_cot['Normalized Prediction'].value_counts()

,count
Normalized Prediction,
B,84
C,63
A,41
D,12


In [ ]:
pred_cot['prompt'] = data3['prompt']
pred_cot.head()

,Predicted,Normalized Prediction,prompt
0,السؤال:\nمقاومتان مقدار كل منهما م إذا وصلتا...,A,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...
1,السؤال:\nما هو مدى الموجات الصوت ةٌ الت تٌمكن ...,B,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...
2,السؤال:\n جسم كتلته 5 كغم وكمية تحركه 15 كغم.م...,C,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...
3,السؤال:\nالفرق في درجات الحرارة الالزمة لمادة ...,B,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...
4,السؤال:\n كمية التحرك للنظام الذي يتكون من كرت...,B,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...


In [ ]:
pred_cot.to_excel('Allam CoT QA Physics.xlsx', index = False)

In [ ]:
print(classification_report(y_true, pred_cot['Normalized Prediction'].values, digits = 4))

              precision    recall  f1-score   support

           A     0.6098    0.5102    0.5556        49
           B     0.4524    0.6552    0.5352        58
           C     0.4286    0.6429    0.5143        42
           D     0.6667    0.1569    0.2540        51

    accuracy                         0.4900       200
   macro avg     0.5393    0.4913    0.4648       200
weighted avg     0.5406    0.4900    0.4641       200

